# eFootball YOLO (6-Class UI) — Colab 免费 T4 GPU 一键训练

- 一键打开（无需配置 GitHub Token，无需挂载 Google Drive）：
  https://colab.research.google.com/github/idhuanxiang/efootball-bot/blob/main/training/colab_train.ipynb
- 说明：Colab 分配免费 GPU 仍需登录一个 Google 账号；本笔记本不挂载 Drive、不写回仓库。
- 建议在菜单 `代码执行程序 -> 更改运行时类型` 中选择 **T4 GPU**，399+ 张图跑 80 epoch 约 **3~5 分钟**。

In [1]:
!nvidia-smi
import pathlib, shutil, zipfile

REPO_DIR = pathlib.Path('/content/efootball-bot')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 https://github.com/idhuanxiang/efootball-bot.git /content/efootball-bot
%cd /content/efootball-bot
!pip -q install -U ultralytics onnx onnxruntime pillow

ds_zip = pathlib.Path('training/dataset.zip')
if ds_zip.is_file() and not (pathlib.Path('training/dataset/images/train')).is_dir():
    with zipfile.ZipFile(ds_zip) as zf:
        zf.extractall('training/dataset')
    print('Unpacked training/dataset.zip')

Fri Nov 27 19:31:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 训练参数（增量续训可把 WEIGHTS 改为上次的 yolo26n_best.pt，FREEZE=10, EPOCHS=30）
EPOCHS = 80
IMGSZ = 640
BATCH = 16
FREEZE = 0
WEIGHTS = 'yolo26n.pt'
ENABLE_UI_AUG = True
SEED = 0

!python training/prepare_dataset.py
if ENABLE_UI_AUG:
    !python training/augment_ui.py --dataset training/dataset --ratio 0.35 --seed {SEED}

[OK] 类别顺序一致（6 类）：player_card / btn_back / btn_confirm / coin_bar / main_menu / item_row
[OK] 标签校验通过：55 帧；待并入 0，内容重复跳过 55，同名已存在跳过 0
     内容重复：neg_matchhub_hub.png, neg_matchhub_user1.png, neg_matchhub_user2.png, neg_weekly_events_1029.png, neg_weekly_events_list.png, neg_weekly_vsplayers.png, new_cap_211946_145.jpg, new_cap_212246_189.jpg, new_cap_213537_367.jpg, new_cap_213549_946.jpg, new_cap_213647_030.jpg, new_cap_214745_619.jpg, new_cap_215045_149.jpg, new_cap_215100_714.jpg, new_cap_215112_474.jpg, new_cap_215229_464.jpg, new_cap_215241_701.jpg, new_cap_215252_877.jpg, new_cap_215302_986.jpg, new_cap_215313_290.jpg, new_cap_220712_292.jpg, new_cap_220723_878.jpg, new_cap_220734_647.jpg, new_cap_220837_904.jpg, new_cap_222137_008.jpg, new_cap_222203_667.jpg, new_cap_222215_855.jpg, new_cap_222226_322.jpg, new_cap_222247_444.jpg, new_dmp_wk_intro_stuck_20261123_041140.jpg, new_sus_014424_066_other_module_page.jpg, new_sus_014528_573_other_module_page.jpg, new_sus_211613_328_train_ti

In [ ]:
!python training/train_yolo.py \
  --weights {WEIGHTS} \
  --epochs {EPOCHS} \
  --imgsz {IMGSZ} \
  --batch {BATCH} \
  --freeze {FREEZE} \
  --seed {SEED} \
  --device 0 \
  --workers 2

[OK] 类别顺序：player_card / btn_back / btn_confirm / coin_bar / main_menu / item_row
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
[OK] 已关闭 Ultralytics 匿名遥测（sync=False）
[OK] 训练用 data.yaml：/content/efootball-bot/training/out/data_train.yaml（path=/content/efootball-bot/training/dataset）
[..] 开始训练：{'data': '/content/efootball-bot/training/out/data_train.yaml', 'epochs': 80, 'imgsz': 640, 'batch': 16, 'workers': 2, 'seed': 0, 'deterministic': False, 'project': '/content/efootball-bot/training/runs', 'name': 'efootball6', 'exist_ok': True, 'fliplr': 0.0, 'flipud': 0.0, 'degrees': 0.0, 'device': '0'}
Ultralytics 8.4.220 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, 

In [ ]:
!python training/verify_model.py \
  --frames training/negatives/images \
  --compare src/EfootballBot/models/yolo/yolo26n.onnx | tee training/out/verify_report.txt

In [ ]:
import pathlib, zipfile
from google.colab import files

bundle = pathlib.Path('training/out/yolo26n_colab_bundle.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in ['training/out/yolo26n_candidate.onnx', 'training/out/yolo26n_best.pt', 'training/out/verify_report.txt']:
        fp = pathlib.Path(p)
        if fp.is_file():
            zf.write(fp, fp.name)
print('Ready to download:', bundle)
files.download(str(bundle))